# Baseline Experimentation: Document Chunking and Embedding Process

This notebook demonstrates how ScholarSync processes academic papers into chunks,
embeds them, and stores them in the Weaviate vector database.

In [2]:
import sys
sys.path.append('..')
from pathlib import Path
import json
from src.ingest import load_pdfs, get_weaviate_client, create_weaviate_schema, WEAVIATE_CLASS_NAME
from src.fetcher import search_arxiv, search_semantic_scholar, download_pdf
import weaviate.classes.query as wq

d:\Agentic_Project\ScholarSync\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Sample Paper Selection

Let's use a sample paper to demonstrate the chunking process.
For this experiment, we'll use a well-known paper on transformers.

In [3]:
# Search for a sample paper
print("Searching for sample paper...")
query = "attention is all you need"
papers = search_arxiv(query, max_results=1)

if papers:
    paper = papers[0]
    print(f"Found paper: {paper.title}")
    print(f"Authors: {', '.join(paper.authors)}")
    print(f"Published: {paper.published}")
    print(f"\nAbstract:\n{paper.abstract[:300]}...")
else:
    print("No papers found")

2025-12-21 21:28:32,999 - INFO - ArXiv search | query='attention is all you need'
2025-12-21 21:28:33,000 - INFO - Requesting page (first: True, try: 0): https://export.arxiv.org/api/query?search_query=attention+is+all+you+need&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=100


Searching for sample paper...


2025-12-21 21:28:42,744 - INFO - Got first page: 100 of 625891 total results
2025-12-21 21:28:42,747 - INFO - ArXiv search done | results=1


Found paper: Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet
Authors: Luke Melas-Kyriazi
Published: 2021-05-06

Abstract:
The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to which attention is responsible for this strong performance remains unclear. In this short report, we ask: is the...


## 2. Download the Paper

Download the PDF to our experiment directory

In [4]:
experiment_dir = Path("../downloaded_papers/baseline_experiment")
experiment_dir.mkdir(parents=True, exist_ok=True)

if papers:
    print(f"\nDownloading PDF...")
    pdf_path = download_pdf(paper, download_dir=experiment_dir)
    print(f"Downloaded to: {pdf_path}")

2025-12-21 21:28:47,815 - INFO - Download skipped (exists) | file=Do_You_Even_Need_Attention_A_Stack_of_Feed-Forward_Layers_Does_Surprisingly_Well_on_ImageNet.pdf



Downloaded to: ..\downloaded_papers\baseline_experiment\Do_You_Even_Need_Attention_A_Stack_of_Feed-Forward_Layers_Does_Surprisingly_Well_on_ImageNet.pdf


## 3. Load and Parse PDF

The `load_pdfs` function extracts text from the PDF and prepares it for chunking

In [5]:
print("\nLoading PDF...")
documents = load_pdfs(specific_files=[Path(pdf_path)])
print(f"Loaded {len(documents)} document(s)")

if documents:
    doc = documents[0]
    print(f"\nDocument metadata:")
    print(f"  - Filename: {doc.metadata.get('filename', 'N/A')}")
    print(f"  - Page count: {doc.metadata.get('page_count', 'N/A')}")
    print(f"\nFirst 500 characters of text:")
    print(doc.text[:500] + "...")

2025-12-21 21:28:52,422 - INFO - Loading PDFs (specific files): 1
2025-12-21 21:28:52,425 - INFO - PDF files to process: 1



Loading PDF...


2025-12-21 21:28:55,847 - INFO - Parsing PDFs with LlamaParse
2025-12-21 21:28:55,851 - INFO - Parsing: Do_You_Even_Need_Attention_A_Stack_of_Feed-Forward_Layers_Does_Surprisingly_Well_on_ImageNet.pdf
2025-12-21 21:29:05,970 - INFO - Documents loaded: 4


Loaded 4 document(s)

Document metadata:
  - Filename: Do_You_Even_Need_Attention_A_Stack_of_Feed-Forward_Layers_Does_Surprisingly_Well_on_ImageNet.pdf
  - Page count: N/A

First 500 characters of text:
# Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet

**Authors:** Luke Melas-Kyriazi
**Affiliation:** Oxford University
**Email:** lukemk@robots.ox.ac.uk

## Abstract

The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to which attention is responsible for this strong performance remains unclear. In this short report, we...


## 4. Hierarchical Chunking Process

ScholarSync uses hierarchical text splitting with multiple levels:
- Level 0: Large chunks (2048 tokens) for broad context
- Level 1: Medium chunks (512 tokens) for balanced retrieval
- Level 2: Small chunks (128 tokens) for precise matching

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from notebook_utils import llama_to_langchain_docs

# Demonstrate chunking at different levels
chunk_sizes = [
    (2048, 200, "Large"),
    (512, 50, "Medium"),  
    (128, 20, "Small")
]

all_chunks = []
for chunk_size, overlap, level_name in chunk_sizes:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    
    if documents:
        # Convert LlamaIndex docs to LangChain format
        langchain_docs = llama_to_langchain_docs(documents)
        
        # Now split the converted documents
        chunks = splitter.split_documents(langchain_docs)
        all_chunks.append((level_name, len(chunks), chunks[:2] if chunks else []))
        print(f"\n{level_name} chunks (size={chunk_size}, overlap={overlap}): {len(chunks)} chunks")


Large chunks (size=2048, overlap=200): 11 chunks

Medium chunks (size=512, overlap=50): 45 chunks

Small chunks (size=128, overlap=20): 188 chunks


## 5. Inspect Sample Chunks

Let's examine the actual content of chunks at different levels

In [10]:
for level_name, count, sample_chunks in all_chunks:
    print(f"\n{'='*60}")
    print(f"{level_name} Chunk Sample ({count} total chunks)")
    print(f"{'='*60}")
    
    if sample_chunks:
        chunk = sample_chunks[0]
        print(f"\nChunk 1 content:")
        print(chunk.page_content[:400] + "..." if len(chunk.page_content) > 400 else chunk.page_content)
        print(f"\nChunk metadata: {chunk.metadata}")


Large Chunk Sample (11 total chunks)

Chunk 1 content:
# Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet

**Authors:** Luke Melas-Kyriazi
**Affiliation:** Oxford University
**Email:** lukemk@robots.ox.ac.uk

## Abstract

The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to ...

Chunk metadata: {'filename': 'Do_You_Even_Need_Attention_A_Stack_of_Feed-Forward_Layers_Does_Surprisingly_Well_on_ImageNet.pdf', 'page': 1, 'creation_date': '2025-12-21T21:29:05.969447', 'parser': 'llamaparse'}

Medium Chunk Sample (45 total chunks)

Chunk 1 content:
# Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet

**Authors:** Luke Melas-Kyriazi
**Affiliation:** Oxford University
**Email:** lukemk@robots.ox.ac.uk

## Abstract

Chunk metadata: {'filename': 'Do_You_Even_Need_Attention_A_

2025-12-21 21:36:10,427 - INFO - Indexing into Weaviate (tenant=baseline_experiment)



Ingesting chunks into Weaviate...
This will create embeddings and store them in the vector database...



2025-12-21 21:36:11,428 - INFO - Nodes: total=42 leaf=27 parent=15
2025-12-21 21:36:11,430 - INFO - Inserting nodes: 42


WeaviateClosedClientError: The `WeaviateClient` is closed. Run `client.connect()` to (re)connect!

## 6. Embedding and Vector Storage

Each chunk is embedded using Gemini embeddings and stored in Weaviate

In [15]:
print("\nConnecting to Weaviate...")
client = get_weaviate_client()
create_weaviate_schema(client)


# Ingest the document chunks into Weaviate
from src.ingest import create_vector_index

print("\nIngesting chunks into Weaviate...")
print("This will create embeddings and store them in the vector database...\n")

# Ingest the documents (this handles chunking and embedding automatically)
stats = create_vector_index(documents, client, tenant_id="baseline_experiment")

print(f"\n✓ Ingestion complete!")
print(f"  - Total nodes inserted: {stats['total_nodes']}")
print(f"  - Leaf nodes: {stats['leaf_nodes']}")
print(f"  - Parent nodes: {stats['parent_nodes']}")

# Check if collection exists
if client.collections.exists(WEAVIATE_CLASS_NAME):
    collection = client.collections.get(WEAVIATE_CLASS_NAME)
    print(f"Collection '{WEAVIATE_CLASS_NAME}' ready")
    
    # Query for chunks from our experiment
    print(f"\nQuerying for chunks from baseline experiment...")
    results = collection.query.fetch_objects(
        filters=wq.Filter.by_property("filename").contains_any([Path(pdf_path).name if pdf_path else ""]),
        limit=5
    )
    
    print(f"\nFound {len(results.objects)} chunks in Weaviate")
    
    if results.objects:
        print("\nSample chunk from Weaviate:")
        obj = results.objects[0]
        print(f"  - Text length: {len(obj.properties.get('text', ''))}")
        print(f"  - Node level: {obj.properties.get('node_level', 'N/A')}")
        print(f"  - Page number: {obj.properties.get('page_number', 'N/A')}")
        print(f"  - Filename: {obj.properties.get('filename', 'N/A')}")
        print(f"\n  Text preview: {obj.properties.get('text', '')[:300]}...")

client.close()


Connecting to Weaviate...


2025-12-21 21:37:11,806 - INFO - Weaviate collection exists: ResearchPaper
2025-12-21 21:37:12,931 - INFO - Indexing into Weaviate (tenant=baseline_experiment)
2025-12-21 21:37:13,006 - INFO - Nodes: total=42 leaf=27 parent=15
2025-12-21 21:37:13,007 - INFO - Inserting nodes: 42



Ingesting chunks into Weaviate...
This will create embeddings and store them in the vector database...



2025-12-21 21:37:29,176 - INFO - Inserted nodes: 42



✓ Ingestion complete!
  - Total nodes inserted: 42
  - Leaf nodes: 27
  - Parent nodes: 15
Collection 'ResearchPaper' ready

Querying for chunks from baseline experiment...

Found 5 chunks in Weaviate

Sample chunk from Weaviate:
  - Text length: 244
  - Node level: 2
  - Page number: 2
  - Filename: Do_You_Even_Need_Attention_A_Stack_of_Feed-Forward_Layers_Does_Surprisingly_Well_on_ImageNet.pdf

  Text preview: These works improve upon the vision transformer architecture, each showing strong performance on ImageNet. However, it is not clear how the different parts of ViT or its many variants contribute to the final performance of each of these models....


## 7. Chunk Statistics Summary

Summary of the chunking process

In [16]:
print("\n" + "="*60)
print("CHUNKING PROCESS SUMMARY")
print("="*60)
print(f"\nPaper: {paper.title if papers else 'N/A'}")
print(f"\nChunking Strategy: Hierarchical with 3 levels")
for level_name, count, _ in all_chunks:
    print(f"  - {level_name}: {count} chunks")

print(f"\nTotal chunks created: {sum(count for _, count, _ in all_chunks)}")
print(f"\nEmbedding Model: Gemini text-embedding-004")
print(f"Vector Database: Weaviate Cloud")
print(f"\nAll chunks are stored with metadata including:")
print(f"  - Filename")
print(f"  - Page number")
print(f"  - Chunk level (for hierarchical retrieval)")
print(f"  - Tenant ID (for multi-user isolation)")


CHUNKING PROCESS SUMMARY

Paper: Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet

Chunking Strategy: Hierarchical with 3 levels
  - Large: 11 chunks
  - Medium: 45 chunks
  - Small: 188 chunks

Total chunks created: 244

Embedding Model: Gemini text-embedding-004
Vector Database: Weaviate Cloud

All chunks are stored with metadata including:
  - Filename
  - Page number
  - Chunk level (for hierarchical retrieval)
  - Tenant ID (for multi-user isolation)


## Key Insights

1. **Hierarchical Chunking**: Using multiple chunk sizes allows the system to
   balance between broad context (large chunks) and precise matching (small chunks)

2. **Metadata Preservation**: Each chunk retains important metadata like page number
   and filename, enabling proper citation in responses

3. **Semantic Search**: Embeddings allow finding relevant content based on meaning,
   not just keyword matching

4. **Scalability**: The multi-tenant design allows multiple users to maintain
   separate paper collections in the same database